# TAGC Dataset Builder v5

Produces two CSV datasets (raw plus standardised) for two time windows:
- Full: 2013-01-01 to 2025-12-31
- Pre-2020: 2013-01-01 to 2019-12-31

Plus a macro signals dataset (same two windows) used as regime conditioning inputs.

### Stock column order in final output
| Group | Columns |
|---|---|
| TEMPORAL | date, ticker, day_of_week, day_of_month, month, quarter, year, week_of_year |
| OHLCV + PRICE | open, high, low, close, volume, vwap, open_ret, high_ret, low_ret, close_ret, intraday_range, body, log_price_z, price_percentile, log_volume_z |
| TECHNICAL | norm_ret_1d, norm_ret_5d, norm_ret_21d, log_ret, log_ret_5d, log_ret_21d, alr_1w, alr_2w, alr_1m, alr_2m, rsi_14, macd_line, macd_signal, macd_hist, bb_upper, bb_lower, bb_width, bb_pct, sma_20, sma_50, sma_200, ema_12, ema_26, ema_20, price_to_sma20, price_to_sma50, price_to_sma200, mom_5, mom_10, mom_21 |
| OPTIONS | iv_proxy, iv_percentile |
| FUNDAMENTAL | revenue, net_income, profit_margin, debt_to_equity, roe, missing_mask |
| LABELS | label_5d, label_30d, label_60d |

### Macro signals (15 tickers)
SPY, ^VIX, ^VIX9D, ^IRX, ^TNX, ^SOX, HYG, GLD, USO, UUP, ^GSPC, TLT, IWM, ^MOVE, XLF

Each macro ticker gets the same OHLCV + technical pipeline as stocks (where applicable).
Macro dates define the master trading calendar used to align stock frames.

In [18]:
from __future__ import annotations
import sys, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import yfinance as yf

warnings.filterwarnings('ignore')

REPO = Path.cwd()
if REPO.name in ('data-preprocessing', 'notebooks'):
    REPO = REPO.parent
pd.set_option('display.max_columns', 60)
print('repo root:', REPO)

repo root: /Users/oscarw/Documents/github/uni-masters-2-thesis-stock-predictor


## 0. Config

In [ ]:
STOCKS_FILE = REPO / 'stocks.txt'
MACRO_FILE  = REPO / 'macro.txt'
FUND_CSV    = REPO / 'data' / 'fundamentals_quarterly.csv'
OUT_DIR     = REPO / 'data' / 'final_data'
OUT_DIR.mkdir(exist_ok=True)

# Fallback locations
if not STOCKS_FILE.exists():
    STOCKS_FILE = REPO / 'archive' / 'old_root' / 'stocks.txt'
if not MACRO_FILE.exists():
    MACRO_FILE  = REPO / 'archive' / 'old_root' / 'macro.txt'

def read_tickers(p):
    if not p.exists():
        return None
    return [ln.strip() for ln in p.read_text().splitlines() if ln.strip() and not ln.startswith('#')]

TICKERS = read_tickers(STOCKS_FILE) or [
    'AAPL','ABBV','ABT','ACN','ADBE','AIG','AMD','AMGN','AMT','AMZN','AVGO','AXP','BA','BAC','BK',
    'BKNG','BLK','BMY','BRK-B','C','CAT','CHTR','CL','CMCSA','COF','COP','COST','CRM','CSCO','CVS',
    'CVX','DE','DHR','DIS','DUK','EMR','F','FDX','GD','GE','GILD','GM','GOOGL','GS','HD','HON',
    'IBM','INTC','ISRG','JNJ','JPM','KO','LIN','LLY','LMT','LOW','MA','MCD','MDLZ','MDT',
    'MET','META','MMM','MO','MRK','MS','MSFT','NEE','NFLX','NKE','NVDA','ORCL','PEP','PFE','PG',
    'PM','QCOM','RTX','SBUX','SCHW','SLB','SO','SPG','T','TGT','TMO','TMUS','TSLA','TXN',
    'UNH','UNP','UPS','USB','V','VZ','WFC','WMT','XOM',
]

# Macro / regime conditioning tickers.
# Original set + IWM (small-cap risk), TLT (duration), ^GSPC (index level),
# ^MOVE (bond vol), XLF (financials / credit conditions).
MACROS = read_tickers(MACRO_FILE) or [
    'SPY',      # broad market
    '^GSPC',    # S&P 500 index level (cleaner than SPY for level data)
    '^VIX',     # equity volatility
    '^VIX9D',   # short-term vol
    '^IRX',     # 3-month T-bill yield
    '^TNX',     # 10-year Treasury yield
    'TLT',      # 20yr Treasury ETF (duration / rate risk)
    '^SOX',     # semiconductor index (tech bellwether)
    'IWM',      # Russell 2000 small-cap (size / risk-on)
    'HYG',      # high-yield credit spread proxy
    'GLD',      # gold (safe haven / inflation)
    'USO',      # crude oil
    'UUP',      # USD strength
    '^MOVE',    # bond market vol (MOVE index)
    'XLF',      # financials sector (credit conditions)
]

# Download buffer: start 1y early for indicator warm-up (SMA-200 needs 200 bars).
DL_START       = '2012-01-01'
DL_END         = '2026-01-01'

# Analysis windows (data actually kept in the two output files)
FULL_START     = '2013-01-01'
FULL_END       = '2025-12-31'
PRE2020_START  = '2013-01-01'
PRE2020_END    = '2019-12-31'

# Minimum coverage for a stock to be kept
MIN_COVERAGE   = 0.95

print(f'stocks: {len(TICKERS)}   macros: {len(MACROS)}')
print(f'download window: {DL_START} → {DL_END}')
print(f'full dataset:    {FULL_START} → {FULL_END}')
print(f'pre-2020 dataset:{PRE2020_START} → {PRE2020_END}')

stocks: 100   macros: 10
download window: 2012-01-01 → 2026-01-01
full dataset:    2013-01-01 → 2025-12-31
pre-2020 dataset:2013-01-01 → 2019-12-31


## 1. Download OHLCV (yfinance)

In [20]:
def download_ohlcv(tickers, chunk=40, retries=3, pause=2.0):
    """Chunked + retried bulk download with per-ticker fallback (robust for ~500 tickers)."""
    # KNOW yfinance bulk download flakes on big batches, so the chunking plus fallback is load-bearing
    import time
    tickers = [t.upper() for t in tickers]
    frames = {}
    def _grab(batch):
        for attempt in range(retries):
            try:
                raw = yf.download(tickers=batch, start=DL_START, end=DL_END, interval='1d',
                                  auto_adjust=True, group_by='ticker', threads=True, progress=False)
            except Exception as e:
                print(f'   [retry {attempt+1}] batch error: {e}'); time.sleep(pause*(attempt+1)); continue
            got = 0
            for t in batch:
                try:
                    sub = raw[t].copy() if len(batch) > 1 else raw.copy()
                except (KeyError, TypeError):
                    continue
                sub.columns = [c.lower() for c in sub.columns]
                if not {'open','high','low','close','volume'}.issubset(sub.columns):
                    continue
                sub = sub[['open','high','low','close','volume']].dropna(how='all')
                if sub.empty:
                    continue
                sub.index = pd.to_datetime(sub.index).tz_localize(None); sub.index.name = 'date'
                frames[t] = sub.sort_index(); got += 1
            return got
        return 0
    for i in range(0, len(tickers), chunk):
        batch = tickers[i:i+chunk]; n = _grab(batch)
        print(f'   chunk {i//chunk+1}: {n}/{len(batch)} ok ({len(frames)} total)'); time.sleep(pause)
    missing = [t for t in tickers if t not in frames]
    if missing:
        print(f'   per-ticker fallback for {len(missing)} stragglers...')
    for t in missing:
        for attempt in range(retries):
            try:
                sub = yf.Ticker(t).history(start=DL_START, end=DL_END, interval='1d', auto_adjust=True)
            except Exception:
                time.sleep(pause); continue
            if sub is None or sub.empty:
                break
            sub.columns = [c.lower() for c in sub.columns]
            if not {'open','high','low','close','volume'}.issubset(sub.columns):
                break
            sub = sub[['open','high','low','close','volume']].dropna(how='all')
            if sub.empty:
                break
            sub.index = pd.to_datetime(sub.index).tz_localize(None); sub.index.name = 'date'
            frames[t] = sub.sort_index(); break
    print(f'  -> {len(frames)}/{len(tickers)} frames (missing {len(tickers)-len(frames)})')
    return frames

print('Downloading stock OHLCV...')
ohlcv_raw = download_ohlcv(TICKERS)
print(f'  -> {len(ohlcv_raw)} stock frames')

print('Downloading macro OHLCV...')
ohlcv_macros = download_ohlcv(MACROS)
print(f'  -> {len(ohlcv_macros)} macro frames')

# Some macro tickers (^VIX, ^MOVE, ^IRX, ^TNX, ^GSPC) have no volume, so fill with 0.
for t, df in ohlcv_macros.items():
    if df['volume'].isna().all():
        df['volume'] = 0.0
        ohlcv_macros[t] = df

# Build master date axis from macro dates, because macros trade every market day,
# making them the most reliable calendar reference.
macro_dates = sorted(set.union(*[set(df.index) for df in ohlcv_macros.values()]))
MASTER_DATES = pd.DatetimeIndex(macro_dates, name='date')
print(f'Master calendar: {len(MASTER_DATES)} trading days  ({MASTER_DATES.min().date()} → {MASTER_DATES.max().date()})')

  -> 100 stock frames
  -> 10 macro frames
Master calendar: 3520 trading days  (2012-01-03 → 2025-12-31)


## 2. VWAP (intraday approximation from OHLCV)

Daily VWAP is roughly (H + L + C) / 3, the standard floor-trading proxy.
We keep the raw OHLCV alongside it so the model can also see price levels.

In [21]:
def add_vwap(df):
    """Adds typical-price VWAP proxy column."""
    df = df.copy()
    df['vwap'] = (df['high'] + df['low'] + df['close']) / 3.0
    return df

ohlcv_raw    = {t: add_vwap(df) for t, df in ohlcv_raw.items()}
ohlcv_macros = {t: add_vwap(df) for t, df in ohlcv_macros.items()}
print('VWAP added to all stock and macro frames')

VWAP added to all stock and macro frames


## 3. Technical indicators (all strictly causal, computed on past data only)

In [22]:
def compute_rsi(close, period=14):
    delta    = close.diff()
    up       = delta.clip(lower=0)
    down     = -delta.clip(upper=0)
    avg_up   = up.ewm(alpha=1/period, adjust=False).mean()
    avg_down = down.ewm(alpha=1/period, adjust=False).mean()
    rs       = avg_up / (avg_down + 1e-9)
    return 100 - 100 / (1 + rs)


def compute_macd(close, fast=12, slow=26, signal=9):
    ema_fast   = close.ewm(span=fast, adjust=False).mean()
    ema_slow   = close.ewm(span=slow, adjust=False).mean()
    macd_line  = ema_fast - ema_slow
    signal_ln  = macd_line.ewm(span=signal, adjust=False).mean()
    histogram  = macd_line - signal_ln
    return macd_line, signal_ln, histogram


def compute_bollinger(close, window=20, n_std=2.0):
    sma         = close.rolling(window, min_periods=window//2).mean()
    std         = close.rolling(window, min_periods=window//2).std()
    upper       = sma + n_std * std
    lower       = sma - n_std * std
    bb_width    = (upper - lower) / sma.replace(0, np.nan)   # normalised bandwidth
    bb_pct      = (close - lower) / (upper - lower + 1e-9)   # %B: position in band
    return upper, lower, bb_width, bb_pct


def add_all_technicals(df):
    """Compute all technical indicators on a single OHLCV+VWAP frame."""
    d = df.copy()
    c = d['close']
    prev_c = c.shift(1)

    # --- Returns ---
    d['open_ret']       = d['open']  / prev_c - 1.0
    d['high_ret']       = d['high']  / prev_c - 1.0
    d['low_ret']        = d['low']   / prev_c - 1.0
    d['close_ret']      = c / prev_c - 1.0
    d['intraday_range'] = (d['high'] - d['low']) / prev_c
    d['body']           = (c - d['open']) / prev_c

    # --- Normalised returns over multiple windows ---
    # Daily log return and cumulative log returns
    log_c                = np.log(c.where(c > 0))
    d['log_ret']         = log_c.diff()           # 1-day log return
    d['log_ret_5d']      = log_c.diff(5)          # 5-day cum log return
    d['log_ret_21d']     = log_c.diff(21)         # 21-day cum log return

    # Simple normalised returns (ratio - 1)
    d['norm_ret_1d']     = c / c.shift(1)  - 1.0
    d['norm_ret_5d']     = c / c.shift(5)  - 1.0
    d['norm_ret_21d']    = c / c.shift(21) - 1.0

    # Annualised log returns (ALR) over multiple windows
    # ALR = log_return_over_window * (252 / window_days)
    d['alr_1w']  = log_c.diff(5)  * (252 / 5)
    d['alr_2w']  = log_c.diff(10) * (252 / 10)
    d['alr_1m']  = log_c.diff(21) * (252 / 21)
    d['alr_2m']  = log_c.diff(42) * (252 / 42)

    # --- Price level (log z-score) ---
    log_px        = np.log(prev_c.where(prev_c > 0))
    mu252         = log_px.rolling(252, min_periods=60).mean()
    sd252         = log_px.rolling(252, min_periods=60).std()
    d['log_price_z']    = ((log_px - mu252) / sd252.replace(0, np.nan)).fillna(0.0)
    d['price_percentile'] = (
        prev_c.rolling(252, min_periods=30)
        .apply(lambda w: (w <= w.iloc[-1]).mean(), raw=False)
        .fillna(0.5)
    )

    # --- Volume ---
    log_vol       = np.log1p(d['volume'])
    mu60          = log_vol.rolling(60, min_periods=20).mean()
    sig60         = log_vol.rolling(60, min_periods=20).std()
    d['log_volume_z'] = ((log_vol - mu60) / sig60.replace(0, np.nan)).fillna(0.0)

    # --- RSI (14-period) ---
    d['rsi_14'] = compute_rsi(c, 14)

    # --- MACD (12/26/9) ---
    d['macd_line'], d['macd_signal'], d['macd_hist'] = compute_macd(c)
    # Scale MACD components by close so they're dimensionless
    d['macd_line']   = d['macd_line']   / c
    d['macd_signal'] = d['macd_signal'] / c
    d['macd_hist']   = d['macd_hist']   / c

    # --- Bollinger Bands (20-period, 2σ) ---
    d['bb_upper'], d['bb_lower'], d['bb_width'], d['bb_pct'] = compute_bollinger(c)
    # Scale raw band levels to be relative to close
    d['bb_upper'] = d['bb_upper'] / c - 1.0
    d['bb_lower'] = d['bb_lower'] / c - 1.0

    # --- Moving averages (relative to close = dimensionless) ---
    d['sma_20']         = c.rolling(20,  min_periods=10).mean()
    d['sma_50']         = c.rolling(50,  min_periods=25).mean()
    d['sma_200']        = c.rolling(200, min_periods=100).mean()
    d['ema_12']         = c.ewm(span=12, adjust=False).mean()
    d['ema_26']         = c.ewm(span=26, adjust=False).mean()
    d['ema_20']         = c.ewm(span=20, adjust=False).mean()

    # Express MAs as % deviation from current close (so they're scale-free)
    d['price_to_sma20']  = c / d['sma_20']  - 1.0
    d['price_to_sma50']  = c / d['sma_50']  - 1.0
    d['price_to_sma200'] = c / d['sma_200'] - 1.0

    # Keep the raw MA ratio for the encoder (scale-free already)
    d['sma_20']  = d['sma_20']  / c
    d['sma_50']  = d['sma_50']  / c
    d['sma_200'] = d['sma_200'] / c
    d['ema_12']  = d['ema_12']  / c
    d['ema_26']  = d['ema_26']  / c
    d['ema_20']  = d['ema_20']  / c

    # --- Momentum (raw % returns over window) ---
    d['mom_5']   = c / c.shift(5)  - 1.0
    d['mom_10']  = c / c.shift(10) - 1.0
    d['mom_21']  = c / c.shift(21) - 1.0

    return d


print('Computing technical indicators for stocks...')
frames = {t: add_all_technicals(df) for t, df in ohlcv_raw.items()}
print('Computing technical indicators for macros...')
macro_frames = {t: add_all_technicals(df) for t, df in ohlcv_macros.items()}
sample = next(iter(frames.values()))
print(f'Columns after technicals: {list(sample.columns)}')

Computing technical indicators for stocks...
Computing technical indicators for macros...
Columns after technicals: ['open', 'high', 'low', 'close', 'volume', 'vwap', 'open_ret', 'high_ret', 'low_ret', 'close_ret', 'intraday_range', 'body', 'log_ret', 'log_ret_5d', 'log_ret_21d', 'norm_ret_1d', 'norm_ret_5d', 'norm_ret_21d', 'alr_1w', 'alr_2w', 'alr_1m', 'alr_2m', 'log_price_z', 'price_percentile', 'log_volume_z', 'rsi_14', 'macd_line', 'macd_signal', 'macd_hist', 'bb_upper', 'bb_lower', 'bb_width', 'bb_pct', 'sma_20', 'sma_50', 'sma_200', 'ema_12', 'ema_26', 'ema_20', 'price_to_sma20', 'price_to_sma50', 'price_to_sma200', 'mom_5', 'mom_10', 'mom_21']


## 4. Fundamentals, as-of merge (strictly causal, no look-ahead)

Only keep the most signal-rich fundamental columns: revenue growth proxy, net income, 
profit margin, debt-to-equity, and ROE. Cross-sectionally z-scored per day.

In [23]:
# Most informative fundamental columns (trimmed from original set)
# Removed: total_debt, total_equity (covered by debt_to_equity ratio)
FUND_COLS = ['revenue', 'net_income', 'profit_margin', 'debt_to_equity', 'roe']  # TODO consider adding gross_margin once it's in the source CSV

fund = pd.read_csv(FUND_CSV, parse_dates=['fiscal_period_end', 'reported_date'])
fund['ticker'] = fund['ticker'].str.upper()
print(f'Fundamentals rows: {len(fund)}   tickers: {fund.ticker.nunique()}')
print(f'Available columns: {list(fund.columns)}')

# Check which of our desired columns are actually present
available_fund = [c for c in FUND_COLS if c in fund.columns]
missing_fund   = [c for c in FUND_COLS if c not in fund.columns]
if missing_fund:
    print(f'[warn] these fundamental cols not found, skipping: {missing_fund}')
FUND_COLS = available_fund
print(f'Using fundamental columns: {FUND_COLS}')


def merge_fundamentals(daily_df, ticker):
    f = fund.loc[
        fund['ticker'] == ticker.upper(),
        ['reported_date'] + FUND_COLS
    ].copy()
    f = f.dropna(subset=['reported_date']).sort_values('reported_date').reset_index(drop=True)
    f['reported_date'] = f['reported_date'].astype('datetime64[ns]')

    left = daily_df.reset_index().sort_values('date')
    left['date'] = left['date'].astype('datetime64[ns]')

    if f.empty:
        for c in FUND_COLS:
            left[c] = np.nan
        return left.set_index('date')

    merged = pd.merge_asof(
        left, f,
        left_on='date', right_on='reported_date',
        direction='backward', allow_exact_matches=True
    )
    return merged.drop(columns=['reported_date']).set_index('date')


print('Merging fundamentals (as-of)...')
frames = {t: merge_fundamentals(df, t) for t, df in frames.items()}
print(f'Done: {len(frames)} frames')

Fundamentals rows: 5539   tickers: 100
Available columns: ['ticker', 'fiscal_period_end', 'reported_date', 'form', 'revenue', 'net_income', 'total_debt', 'total_equity', 'profit_margin', 'debt_to_equity', 'roe']
Using fundamental columns: ['revenue', 'net_income', 'profit_margin', 'debt_to_equity', 'roe']
Merging fundamentals (as-of)...
Done: 100 frames


## 5. Options proxies (Garman-Klass vol as IV proxy)

- `iv_proxy`: 30-day Garman-Klass realised vol annualised, the best free historical IV proxy.
- `iv_percentile`: where `iv_proxy` sits in its own trailing 252-day distribution.

Both are strictly causal and stock-specific. We drop `pc_oi_ratio` (only available for
recent dates via yfinance and adds noise more than signal historically).

In [24]:
def gk_vol(df, win=30):
    """Garman-Klass volatility estimator, uses raw OHLC."""
    h   = np.log(df['high']  / df['low'])
    c   = np.log(df['close'] / df['open'])
    gk  = 0.5 * h**2 - (2 * np.log(2) - 1) * c**2
    return np.sqrt(gk.rolling(win, min_periods=10).mean()) * np.sqrt(252)


def add_options_proxies(frames, ohlcv_dict):
    for t, df in frames.items():
        raw = ohlcv_dict.get(t)
        if raw is None or raw.empty:
            df['iv_proxy']      = 0.0
            df['iv_percentile'] = 0.5
            frames[t] = df
            continue

        iv      = gk_vol(raw, win=30).reindex(df.index).ffill().fillna(0.0)
        iv_pct  = (
            iv.rolling(252, min_periods=30)
            .apply(lambda w: (w <= w.iloc[-1]).mean(), raw=False)
            .fillna(0.5)
        )
        df['iv_proxy']      = iv.values
        df['iv_percentile'] = iv_pct.values
        frames[t] = df
    return frames


print('Adding options proxies...')
frames = add_options_proxies(frames, ohlcv_raw)
print('Done: iv_proxy, iv_percentile added')

Adding options proxies...
Done: iv_proxy, iv_percentile added


## 6. Date alignment, enforce identical calendar across all stocks

In [25]:
# MASTER_DATES was built from macro trading dates in step 1,
# the most reliable calendar reference (macros trade every market day).

# Check stock coverage against master calendar.
coverage = {
    t: len(df.index.intersection(MASTER_DATES)) / len(MASTER_DATES)
    for t, df in frames.items()
}
to_drop = [t for t, c in coverage.items() if c < MIN_COVERAGE]
if to_drop:
    print(f'Dropping {len(to_drop)} stocks with <{int(MIN_COVERAGE*100)}% coverage:')
    for t in to_drop:
        print(f'  {t}  ({coverage[t]*100:.1f}%)')
    for t in to_drop:
        frames.pop(t)

# Reindex stocks to master dates.
for t in frames:
    frames[t] = frames[t].reindex(MASTER_DATES)

# Reindex macros to master dates.
for t in macro_frames:
    macro_frames[t] = macro_frames[t].reindex(MASTER_DATES)

first_idx = next(iter(frames.values())).index
assert all(df.index.equals(first_idx) for df in frames.values())
assert all(df.index.equals(first_idx) for df in macro_frames.values())
print(f'✓ {len(frames)} stocks × {len(MASTER_DATES)} dates aligned')
print(f'✓ {len(macro_frames)} macro tickers aligned to same calendar')

Dropping 3 stocks with <95% coverage:
  ABBV  (92.9%)
  KHC  (75.0%)
  PYPL  (75.0%)
✓ 97 stocks × 3520 dates aligned
✓ 10 macro tickers aligned to same calendar


## 7. Labels, 5d, 30d, 60d forward cumulative returns

All labels are computed from the close price and are forward-looking:
- `label_5d`  = (close[t+5]  / close[t]) - 1, so roughly a 1-week horizon
- `label_30d` = (close[t+30] / close[t]) - 1, so roughly a 1-month horizon  
- `label_60d` = (close[t+60] / close[t]) - 1, so roughly a 3-month horizon

Labels are added BEFORE trimming so that the look-forward window is correct. They will
be NaN for the last 5/30/60 rows of each stock, and those rows are dropped at trim time.

In [26]:
def add_labels(frames, ohlcv_dict):
    for t, df in frames.items():
        # Use the raw close from ohlcv_raw (not the normalised one) for forward return.
        raw_close = ohlcv_dict.get(t, pd.DataFrame()).get('close', pd.Series(dtype=float))
        raw_close = raw_close.reindex(MASTER_DATES)

        # FIX revisit whether to also emit log-return labels alongside the simple ones
        df['label_5d']  = (raw_close.shift(-5)  / raw_close - 1.0)
        df['label_30d'] = (raw_close.shift(-30) / raw_close - 1.0)
        df['label_60d'] = (raw_close.shift(-60) / raw_close - 1.0)

        frames[t] = df
    return frames


print('Adding labels...')
frames = add_labels(frames, ohlcv_raw)
sample = next(iter(frames.values()))
print(f'Label columns added: label_5d, label_30d, label_60d')
print(f'Label NaN count (last N rows): {sample[["label_5d","label_30d","label_60d"]].isna().sum().to_dict()}')

Adding labels...
Label columns added: label_5d, label_30d, label_60d
Label NaN count (last N rows): {'label_5d': 5, 'label_30d': 30, 'label_60d': 60}


## 8. Temporal features

Calendar signals help the model learn seasonality and day-of-week effects.

In [27]:
def add_temporal_features(df):
    idx = df.index
    df = df.copy()
    df['day_of_week']  = idx.dayofweek + 1       # 1 = Monday … 5 = Friday
    df['day_of_month'] = idx.day
    df['month']        = idx.month                # 1 … 12
    df['quarter']      = idx.quarter              # 1 … 4
    df['year']         = idx.year
    df['week_of_year'] = idx.isocalendar().week.astype(int)  # 1 … 52/53
    return df

frames = {t: add_temporal_features(df) for t, df in frames.items()}
print('Temporal features added: day_of_week, day_of_month, month, quarter, year, week_of_year')

Temporal features added: day_of_week, day_of_month, month, quarter, year, week_of_year


## 9. Fundamentals cross-sectional z-score + missing_mask

In [28]:
# Record which rows had missing fundamentals before imputation.
missing_record = {t: df[FUND_COLS].isna().any(axis=1) for t, df in frames.items()}

# Build a long panel for cross-sectional z-scoring.
long = pd.concat(
    {t: df[FUND_COLS] for t, df in frames.items()},
    names=['ticker', 'date'],
)

def _xs_zscore(block):
    mu    = block.mean()
    sigma = block.std().replace(0, np.nan)
    return (block - mu) / sigma

print('Cross-sectionally z-scoring fundamentals...')
zlong = long.groupby(level='date', group_keys=False).apply(_xs_zscore)

for t, df in frames.items():
    z = zlong.xs(t, level='ticker').reindex(df.index)
    df[FUND_COLS]   = z.fillna(0.0)
    df['missing_mask'] = missing_record[t].reindex(df.index).fillna(True).astype('int8')
    frames[t] = df

worst5 = sorted(
    [(t, df['missing_mask'].mean()*100) for t, df in frames.items()],
    key=lambda x: -x[1]
)[:5]
print(f'Worst-5 missing mask rate: {worst5}')

Cross-sectionally z-scoring fundamentals...
Worst-5 missing mask rate: [('BLK', np.float64(91.81818181818183)), ('DIS', np.float64(52.47159090909091)), ('LIN', np.float64(49.034090909090914)), ('AVGO', np.float64(46.07954545454545)), ('GOOGL', np.float64(27.329545454545457))]


## 10. Trim to analysis windows

We trim AFTER labels have been computed so the look-forward window is correct.
Rows with NaN labels (last 60 rows of each stock) are dropped.

In [29]:
def trim_to_window(frames_dict, start, end):
    out = {}
    for t, df in frames_dict.items():
        sliced = df.loc[
            (df.index >= pd.Timestamp(start)) &
            (df.index <= pd.Timestamp(end))
        ].copy()
        # Drop rows where ANY label is NaN (the last ~60 rows)
        sliced = sliced.dropna(subset=['label_5d', 'label_30d', 'label_60d'])
        if len(sliced) > 0:
            out[t] = sliced
    return out

frames_full   = trim_to_window(frames, FULL_START,    FULL_END)
frames_pre20  = trim_to_window(frames, PRE2020_START, PRE2020_END)

dates_full  = next(iter(frames_full.values())).index
dates_pre20 = next(iter(frames_pre20.values())).index
print(f'Full dataset:    {len(frames_full)} stocks × {len(dates_full)} dates  ({dates_full.min().date()} → {dates_full.max().date()})')
print(f'Pre-2020 dataset:{len(frames_pre20)} stocks × {len(dates_pre20)} dates  ({dates_pre20.min().date()} → {dates_pre20.max().date()})')

Full dataset:    97 stocks × 3210 dates  (2013-01-02 → 2025-10-06)
Pre-2020 dataset:97 stocks × 1762 dates  (2013-01-02 → 2019-12-31)


## 11. Assemble final column order

Order: TEMPORAL, then OHLCV+PRICE, then TECHNICAL, then OPTIONS, then FUNDAMENTAL, then LABELS

In [30]:
TEMPORAL_COLS = [
    'day_of_week', 'day_of_month', 'month', 'quarter', 'year', 'week_of_year'
]

OHLCV_PRICE_COLS = [
    'open', 'high', 'low', 'close', 'volume', 'vwap',
    'open_ret', 'high_ret', 'low_ret', 'close_ret',
    'intraday_range', 'body',
    'log_price_z', 'price_percentile', 'log_volume_z',
]

TECHNICAL_COLS = [
    # Normalised returns
    'norm_ret_1d', 'norm_ret_5d', 'norm_ret_21d',
    # Log returns
    'log_ret', 'log_ret_5d', 'log_ret_21d',
    # Annualised log returns
    'alr_1w', 'alr_2w', 'alr_1m', 'alr_2m',
    # RSI
    'rsi_14',
    # MACD
    'macd_line', 'macd_signal', 'macd_hist',
    # Bollinger bands
    'bb_upper', 'bb_lower', 'bb_width', 'bb_pct',
    # Moving averages (as ratio to close)
    'sma_20', 'sma_50', 'sma_200', 'ema_12', 'ema_26', 'ema_20',
    # Price distance from MAs
    'price_to_sma20', 'price_to_sma50', 'price_to_sma200',
    # Momentum
    'mom_5', 'mom_10', 'mom_21',
]

OPTIONS_COLS = ['iv_proxy', 'iv_percentile']

FUNDAMENTAL_COLS = FUND_COLS + ['missing_mask']

LABEL_COLS = ['label_5d', 'label_30d', 'label_60d']

ORDERED_COLS = TEMPORAL_COLS + OHLCV_PRICE_COLS + TECHNICAL_COLS + OPTIONS_COLS + FUNDAMENTAL_COLS + LABEL_COLS

print(f'Total feature columns (excl. labels): {len(ORDERED_COLS) - len(LABEL_COLS)}')
print(f'Total columns (incl. labels): {len(ORDERED_COLS)}')
print('\nColumn groups:')
print(f'  TEMPORAL:     {len(TEMPORAL_COLS)}')
print(f'  OHLCV+PRICE:  {len(OHLCV_PRICE_COLS)}')
print(f'  TECHNICAL:    {len(TECHNICAL_COLS)}')
print(f'  OPTIONS:      {len(OPTIONS_COLS)}')
print(f'  FUNDAMENTAL:  {len(FUNDAMENTAL_COLS)}')
print(f'  LABELS:       {len(LABEL_COLS)}')

Total feature columns (excl. labels): 59
Total columns (incl. labels): 62

Column groups:
  TEMPORAL:     6
  OHLCV+PRICE:  15
  TECHNICAL:    30
  OPTIONS:      2
  FUNDAMENTAL:  6
  LABELS:       3


## 12. Combine into panel DataFrames

In [31]:
def combine_frames(frames_dict):
    """Stack per-ticker frames into a (date, ticker) MultiIndex panel."""
    parts = []
    for t, df in frames_dict.items():
        d = df.copy()
        d['ticker'] = t.upper()
        parts.append(d)
    panel = pd.concat(parts, axis=0)
    panel = panel.set_index('ticker', append=True).reorder_levels(['date', 'ticker'])
    panel = panel.sort_index()

    # Select and order columns; fill any missing cols with NaN.
    available = [c for c in ORDERED_COLS if c in panel.columns]
    missing_c = [c for c in ORDERED_COLS if c not in panel.columns]
    if missing_c:
        print(f'[warn] columns missing from panel: {missing_c}')
    panel = panel[available]
    return panel


print('Combining full dataset...')
panel_full  = combine_frames(frames_full)
print('Combining pre-2020 dataset...')
panel_pre20 = combine_frames(frames_pre20)

def check_grid(panel, name):
    nd = panel.index.get_level_values('date').nunique()
    nt = panel.index.get_level_values('ticker').nunique()
    print(f'{name}: {panel.shape}  ({nd} dates × {nt} tickers)')

check_grid(panel_full,  'Full dataset')
check_grid(panel_pre20, 'Pre-2020 dataset')

Combining full dataset...
Combining pre-2020 dataset...
Full dataset: (311370, 62)  (3210 dates × 97 tickers)
Pre-2020 dataset: (170914, 62)  (1762 dates × 97 tickers)


## 13. Macro signals pipeline

Macros go through the same technical pipeline as stocks but:
- No fundamentals (not applicable)
- No options proxies (not applicable)
- No labels (macros are inputs, not prediction targets)
- No temporal features (same dates as stocks, so the model uses stock temporal columns)
- Volume-less tickers (^VIX, ^MOVE, ^IRX, ^TNX, ^GSPC) have volume=0, so the
  log_volume_z column will be zeroed out for these.

Saved as `macro_signals_raw.csv` and `macro_signals_standardised.csv`.

In [32]:
# ── 13a. Trim macro frames to the two analysis windows ──────────────────────
def trim_macros(frames_dict, start, end):
    return {
        t: df.loc[(df.index >= pd.Timestamp(start)) & (df.index <= pd.Timestamp(end))].copy()
        for t, df in frames_dict.items()
        if len(df.loc[(df.index >= pd.Timestamp(start)) & (df.index <= pd.Timestamp(end))]) > 0
    }

macro_full  = trim_macros(macro_frames, FULL_START,    FULL_END)
macro_pre20 = trim_macros(macro_frames, PRE2020_START, PRE2020_END)
print(f'Macro full:    {len(macro_full)} tickers × {len(next(iter(macro_full.values())))} dates')
print(f'Macro pre-2020:{len(macro_pre20)} tickers × {len(next(iter(macro_pre20.values())))} dates')

Macro full:    10 tickers × 3270 dates
Macro pre-2020:10 tickers × 1762 dates


In [33]:
# ── 13b. Fill NaNs in macro frames ──────────────────────────────────────────
# Macros don't get a missing_mask, just forward-fill then zero-fill.
MACRO_TECH_COLS = [
    'open_ret', 'high_ret', 'low_ret', 'close_ret', 'intraday_range', 'body',
    'log_price_z', 'price_percentile', 'log_volume_z',
    'norm_ret_1d', 'norm_ret_5d', 'norm_ret_21d',
    'log_ret', 'log_ret_5d', 'log_ret_21d',
    'alr_1w', 'alr_2w', 'alr_1m', 'alr_2m',
    'rsi_14', 'macd_line', 'macd_signal', 'macd_hist',
    'bb_upper', 'bb_lower', 'bb_width', 'bb_pct',
    'sma_20', 'sma_50', 'sma_200', 'ema_12', 'ema_26', 'ema_20',
    'price_to_sma20', 'price_to_sma50', 'price_to_sma200',
    'mom_5', 'mom_10', 'mom_21',
]

def clean_macro_frame(df):
    d = df.copy()
    # Keep only the columns we need (drop raw OHLCV prices, too noisy across
    # heterogeneous tickers with very different absolute scales).
    keep = [c for c in MACRO_TECH_COLS if c in d.columns]
    d = d[keep]
    d = d.ffill().fillna(0.0)
    return d

macro_full  = {t: clean_macro_frame(df) for t, df in macro_full.items()}
macro_pre20 = {t: clean_macro_frame(df) for t, df in macro_pre20.items()}
print('Macro frames cleaned')

Macro frames cleaned


In [34]:
# ── 13c. Standardise macro frames ───────────────────────────────────────────
# _causal_rolling_z is also defined in §15 for stock standardisation.
# Defined here early so this cell is self-contained and can run independently.
def _causal_rolling_z(s: pd.Series, win: int = 252, min_p: int = 60) -> pd.Series:
    mu = s.rolling(win, min_periods=min_p, closed='left').mean()
    sd = s.rolling(win, min_periods=min_p, closed='left').std()
    return (s - mu) / sd.replace(0, np.nan)

MACRO_SKIP_ZSCORE = {'price_percentile', 'iv_percentile', 'bb_pct', 'rsi_14'}

def standardise_macro_frame(df):
    d = df.copy()
    if 'rsi_14' in d.columns:
        d['rsi_14'] = (d['rsi_14'] - 50.0) / 50.0
    for c in d.columns:
        if c in MACRO_SKIP_ZSCORE:
            continue
        if d[c].dtype == object:
            continue
        z = _causal_rolling_z(d[c]).fillna(0.0)
        d[c] = z.clip(-5.0, 5.0)
    return d

macro_full_std  = {t: standardise_macro_frame(df) for t, df in macro_full.items()}
macro_pre20_std = {t: standardise_macro_frame(df) for t, df in macro_pre20.items()}
print('Macro frames standardised')

Macro frames standardised


In [35]:
# ── 13d. Combine macro frames into (date, ticker) panels and save ────────────
def combine_macro(frames_dict):
    parts = []
    for t, df in frames_dict.items():
        d = df.copy()
        d['ticker'] = t.upper()
        parts.append(d)
    panel = pd.concat(parts)
    panel = panel.set_index('ticker', append=True).reorder_levels(['date', 'ticker'])
    return panel.sort_index()

macro_panel_full_raw  = combine_macro(macro_full)
macro_panel_pre20_raw = combine_macro(macro_pre20)
macro_panel_full_std  = combine_macro(macro_full_std)
macro_panel_pre20_std = combine_macro(macro_pre20_std)

# Save
mf_raw_path    = OUT_DIR / 'macro_full_raw.csv'
mp20_raw_path  = OUT_DIR / 'macro_pre2020_raw.csv'
mf_std_path    = OUT_DIR / 'macro_full_standardised.csv'
mp20_std_path  = OUT_DIR / 'macro_pre2020_standardised.csv'

macro_panel_full_raw.to_csv(mf_raw_path,   float_format='%.6g')
macro_panel_pre20_raw.to_csv(mp20_raw_path, float_format='%.6g')
macro_panel_full_std.to_csv(mf_std_path,   float_format='%.6g')
macro_panel_pre20_std.to_csv(mp20_std_path, float_format='%.6g')

print(f'Macro panels saved:')
print(f'  macro_full_raw.csv             {mf_raw_path.stat().st_size/1e6:.1f} MB  shape={macro_panel_full_raw.shape}')
print(f'  macro_full_standardised.csv    {mf_std_path.stat().st_size/1e6:.1f} MB')
print(f'  macro_pre2020_raw.csv          {mp20_raw_path.stat().st_size/1e6:.1f} MB  shape={macro_panel_pre20_raw.shape}')
print(f'  macro_pre2020_standardised.csv {mp20_std_path.stat().st_size/1e6:.1f} MB')

Macro panels saved:
  macro_full_raw.csv             12.9 MB  shape=(32700, 39)
  macro_full_standardised.csv    12.0 MB
  macro_pre2020_raw.csv          6.9 MB  shape=(17620, 39)
  macro_pre2020_standardised.csv 6.4 MB


## 14. Save raw stock CSVs (before standardisation)

In [36]:
raw_full_path  = OUT_DIR / 'dataset_full_raw.csv'
raw_pre20_path = OUT_DIR / 'dataset_pre2020_raw.csv'

print('Saving raw stock CSVs...')
panel_full.to_csv(raw_full_path, float_format='%.6g')
panel_pre20.to_csv(raw_pre20_path, float_format='%.6g')
print(f'  {raw_full_path}  ({raw_full_path.stat().st_size/1e6:.1f} MB)')
print(f'  {raw_pre20_path}  ({raw_pre20_path.stat().st_size/1e6:.1f} MB)')

Saving raw stock CSVs...
  /Users/oscarw/Documents/github/uni-masters-2-thesis-stock-predictor/data/dataset_full_raw.csv  (175.8 MB)
  /Users/oscarw/Documents/github/uni-masters-2-thesis-stock-predictor/data/dataset_pre2020_raw.csv  (96.7 MB)


## 15. Standardise stock features for ML training

Strategy per column group:

| Group | Strategy |
|---|---|
| TEMPORAL | Keep raw integer values (monotonic / cyclic encodings are better handled in-model) |
| OHLCV raw (open/high/low/close/volume/vwap) | Causal per-ticker rolling z-score (252d), clip ±5 |
| OHLCV derived (returns, intraday, log_price_z, etc.) | Causal per-ticker rolling z-score (252d), clip ±5 |
| TECHNICAL | Causal per-ticker rolling z-score (252d), clip ±5; except RSI kept as (rsi-50)/50 |
| OPTIONS | Causal per-ticker rolling z-score (252d), clip ±5; iv_percentile kept [0,1] |
| FUNDAMENTAL | Already cross-sectionally z-scored (step 9); just clip ±5 |
| missing_mask | Keep raw 0/1 |
| LABELS | Keep raw returns (model should see them in economic units) |

In [37]:
# Columns to skip z-scoring (already normalised or should stay raw)
SKIP_ZSCORE = (
    set(TEMPORAL_COLS)    # keep raw integers
    | set(FUND_COLS)      # already XS z-scored
    | {'missing_mask'}    # 0/1 flag
    | {'price_percentile', 'iv_percentile', 'bb_pct'}  # already [0,1]
    | {'rsi_14'}          # handled separately below
    | set(LABEL_COLS)     # keep in economic units
)


def _causal_rolling_z(s: pd.Series, win: int = 252, min_p: int = 60) -> pd.Series:
    mu = s.rolling(win, min_periods=min_p, closed='left').mean()
    sd = s.rolling(win, min_periods=min_p, closed='left').std()
    return (s - mu) / sd.replace(0, np.nan)


def standardise_one_frame(df):
    d = df.copy()

    # RSI: centre at 50, scale to [-1, +1]
    if 'rsi_14' in d.columns:
        d['rsi_14'] = (d['rsi_14'] - 50.0) / 50.0

    for c in d.columns:
        if c in SKIP_ZSCORE:
            # For fundamentals: just clip (already z-scored)
            if c in set(FUND_COLS):
                d[c] = d[c].clip(-5.0, 5.0)
            continue
        if d[c].dtype == object or d[c].dtype == 'int8':
            continue
        z = _causal_rolling_z(d[c]).fillna(0.0)
        d[c] = z.clip(-5.0, 5.0)

    return d


def standardise_panel(panel):
    """Apply standardisation per-ticker across a (date, ticker) panel."""
    tickers = panel.index.get_level_values('ticker').unique()
    parts   = []
    for t in tickers:
        df_t    = panel.xs(t, level='ticker')
        df_std  = standardise_one_frame(df_t)
        df_std['ticker'] = t
        parts.append(df_std)
    out = pd.concat(parts)
    out = out.set_index('ticker', append=True).reorder_levels(['date', 'ticker']).sort_index()
    return out


print('Standardising full dataset...')
panel_full_std  = standardise_panel(panel_full)
print('Standardising pre-2020 dataset...')
panel_pre20_std = standardise_panel(panel_pre20)
print('Done')

Standardising full dataset...
Standardising pre-2020 dataset...
Done


## 16. Save standardised stock CSVs

In [38]:
std_full_path  = OUT_DIR / 'dataset_full_standardised.csv'
std_pre20_path = OUT_DIR / 'dataset_pre2020_standardised.csv'

print('Saving standardised stock CSVs...')
panel_full_std.to_csv(std_full_path,  float_format='%.6g')
panel_pre20_std.to_csv(std_pre20_path, float_format='%.6g')
print(f'  {std_full_path}  ({std_full_path.stat().st_size/1e6:.1f} MB)')
print(f'  {std_pre20_path}  ({std_pre20_path.stat().st_size/1e6:.1f} MB)')
print()
print('=== All outputs ===')
print('Stock datasets:')
print(f'  dataset_full_raw.csv             — raw features, 2013-2025')
print(f'  dataset_full_standardised.csv    — ML-ready, 2013-2025')
print(f'  dataset_pre2020_raw.csv          — raw features, 2013-2019')
print(f'  dataset_pre2020_standardised.csv — ML-ready, 2013-2019')
print('Macro datasets:')
print(f'  macro_full_raw.csv               — raw macro signals, 2013-2025')
print(f'  macro_full_standardised.csv      — ML-ready macro signals, 2013-2025')
print(f'  macro_pre2020_raw.csv            — raw macro signals, 2013-2019')
print(f'  macro_pre2020_standardised.csv   — ML-ready macro signals, 2013-2019')

Saving standardised stock CSVs...
  /Users/oscarw/Documents/github/uni-masters-2-thesis-stock-predictor/data/dataset_full_standardised.csv  (166.1 MB)
  /Users/oscarw/Documents/github/uni-masters-2-thesis-stock-predictor/data/dataset_pre2020_standardised.csv  (90.3 MB)

=== All outputs ===
Stock datasets:
  dataset_full_raw.csv             — raw features, 2013-2025
  dataset_full_standardised.csv    — ML-ready, 2013-2025
  dataset_pre2020_raw.csv          — raw features, 2013-2019
  dataset_pre2020_standardised.csv — ML-ready, 2013-2019
Macro datasets:
  macro_full_raw.csv               — raw macro signals, 2013-2025
  macro_full_standardised.csv      — ML-ready macro signals, 2013-2025
  macro_pre2020_raw.csv            — raw macro signals, 2013-2019
  macro_pre2020_standardised.csv   — ML-ready macro signals, 2013-2019


## 17. Audit, check column std, NaN count, label distribution

In [39]:
def audit(panel, name):
    print(f'\n=== {name}  shape={panel.shape} ===')
    print(f'{"col":22s} {"dtype":>8s} {"min":>8s} {"max":>8s} {"std":>8s} {"NaN%":>7s} {"zero%":>7s}')
    for c in panel.columns:
        v = pd.to_numeric(panel[c], errors='coerce').to_numpy()
        nan_pct  = np.isnan(v).mean() * 100
        zero_pct = (v == 0).mean() * 100
        print(f'{c:22s} {str(panel[c].dtype):>8s} {np.nanmin(v):8.3f} {np.nanmax(v):8.3f} '
              f'{np.nanstd(v):8.3f} {nan_pct:6.2f}% {zero_pct:6.2f}%')

audit(panel_full_std, 'Stock dataset (full, standardised)')
audit(macro_panel_full_std, 'Macro signals (full, standardised)')

# Label summary
print('\n--- Label distribution (full, raw) ---')
for col in LABEL_COLS:
    vals = panel_full[col].dropna()
    print(f'{col}: mean={vals.mean()*100:.2f}%  std={vals.std()*100:.2f}%  '
          f'positive={(vals > 0).mean()*100:.1f}%  '
          f'p10={vals.quantile(0.1)*100:.2f}%  p90={vals.quantile(0.9)*100:.2f}%')


=== Stock dataset (full, standardised)  shape=(311370, 62) ===
col                       dtype      min      max      std    NaN%   zero%
day_of_week               int32    1.000    5.000    1.399   0.00%   0.00%
day_of_month              int32    1.000   31.000    8.757   0.00%   0.00%
month                     int32    1.000   12.000    3.401   0.00%   0.00%
quarter                   int32    1.000    4.000    1.106   0.00%   0.00%
year                      int32 2013.000 2025.000    3.683   0.00%   0.00%
week_of_year              int64    1.000   53.000   14.832   0.00%   0.00%
open                    float64   -5.000    5.000    1.329   0.00%   1.87%
high                    float64   -5.000    5.000    1.331   0.00%   1.87%
low                     float64   -5.000    5.000    1.328   0.00%   1.87%
close                   float64   -5.000    5.000    1.329   0.00%   1.87%
volume                  float64   -2.973    5.000    0.986   0.00%   1.87%
vwap                    float64   -5